# TP1 - MFA, RBAC, ABAC et Audit de sécurité
Notebook complet d’analyse du contrôle d’accès hospitalier.

## 1. Import des bibliothèques et connexion MongoDB

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pymongo import MongoClient
sns.set_theme(style='whitegrid', palette='crest')
DATA_DIR = Path('datasets')
try:
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=1500)
    client.admin.command('ping')
    db = client['hopital_db']
    print('Connexion MongoDB OK')
except Exception as exc:
    db = None
    print('MongoDB indisponible, poursuite sur CSV:', exc)


## 2. Chargement et exploration des données

In [ ]:
users = pd.read_csv(DATA_DIR / 'users.csv')
resources = pd.read_csv(DATA_DIR / 'resources.csv')
logs = pd.read_csv(DATA_DIR / 'access_logs.csv')
logs['timestamp'] = pd.to_datetime(logs['timestamp'], errors='coerce')
print('Users:', users.shape, 'Resources:', resources.shape, 'Logs:', logs.shape)
display(users.head(), resources.head(), logs.head())
display(logs.isna().sum())


## 3. Analyse des politiques de sécurité

In [ ]:
policy = {
    'medecin': {'read': ['dossier_medical','resultat_labo'], 'write': ['dossier_medical']},
    'infirmier': {'read': ['dossier_medical','resultat_labo']},
    'secretaire': {'read': ['dossier_admin'], 'write': ['dossier_admin'], 'create': ['dossier_admin']},
    'admin_securite': {'read': ['journal_acces'], 'export': ['journal_acces']},
    'patient': {'read': ['dossier_admin']},
}
resource_types = sorted(set(resources['type']).union({'journal_acces'}))
matrix = pd.DataFrame(0, index=list(policy), columns=resource_types)
for role, actions in policy.items():
    for action, types in actions.items():
        for typ in types: matrix.loc[role, typ] += 1
display(matrix)
plt.figure(figsize=(10,4)); sns.heatmap(matrix, annot=True, cmap='YlGnBu')
plt.title('Matrice RBAC/ABAC: rôle vs type de ressource'); plt.show()
display(pd.DataFrame([
 {'contrainte':'MFA','condition':'sensibilité sensible/confidentiel','objectif':'Confidentialité'},
 {'contrainte':'Département','condition':'médecin/infirmier = owner_department','objectif':'Moindre privilège'},
 {'contrainte':'Horaire','condition':'06h à 20h UTC','objectif':'Réduction du risque'}]))


## 4. Simulation du moteur de décision

In [ ]:
MFA_REQUIRED={'sensible','confidentiel'}; DEPT_RESTRICTED={'medecin','infirmier'}
def decide(user, resource, action='read', mfa_ok=True, hour=10):
    if resource['sensitivity'] in MFA_REQUIRED and not mfa_ok: return False, 'MFA obligatoire'
    if resource['type'] not in policy.get(user['role'], {}).get(action, []): return False, 'RBAC: type/action non autorisé'
    if user['role'] in DEPT_RESTRICTED and user['department'] != resource['owner_department']: return False, 'ABAC: département différent'
    if hour < 6 or hour > 20: return False, 'Accès hors horaire'
    return True, 'Accès autorisé'
def U(role, dept='cardiologie', user_id='uX'): return {'role': role, 'department': dept, 'user_id': user_id}
def R(typ, dept='cardiologie', sens='interne', pid='p001'): return {'type': typ, 'owner_department': dept, 'sensitivity': sens, 'patient_id': pid}
tests=[('Cas 1 Médecin service',U('medecin','cardiologie'),R('dossier_medical','cardiologie'),True,True),('Cas 2 Médecin autre service',U('medecin','cardiologie'),R('dossier_medical','pediatrie'),True,False),('Cas 3 Infirmier service',U('infirmier','urgence'),R('resultat_labo','urgence'),True,True),('Cas 4 Infirmier autre service',U('infirmier','urgence'),R('resultat_labo','cardiologie'),True,False),('Cas 5 Secrétaire admin',U('secretaire','accueil'),R('dossier_admin','cardiologie'),True,True),('Cas 6 Secrétaire médical',U('secretaire','accueil'),R('dossier_medical','cardiologie'),True,False),('Cas 7 Patient propre dossier',U('patient','public','p001'),R('dossier_admin','public','interne','p001'),True,True),('Cas 8 Patient autre dossier',U('patient','public','p001'),R('dossier_medical','public','interne','p002'),True,False),('Cas 9 Médecin sensible sans MFA',U('medecin','cardiologie'),R('dossier_medical','cardiologie','sensible'),False,False),('Cas 10 Médecin sensible avec MFA',U('medecin','cardiologie'),R('dossier_medical','cardiologie','sensible'),True,True)]
rows=[]
for name,u,r,mfa,expected in tests:
    ok,reason=decide(u,r,mfa_ok=mfa); rows.append({'cas':name,'attendu':expected,'obtenu':ok,'résultat':'OK' if ok==expected else 'ECHEC','raison':reason})
results=pd.DataFrame(rows); display(results); assert (results['résultat']=='OK').all()


## 5. Analyse des logs d’audit

In [ ]:
logs['decision']=np.where(logs['success'].astype(str).str.lower().eq('true'),'ALLOW','DENY')
logs['hour']=logs['timestamp'].dt.hour
display(logs.groupby(['decision','user_id']).size().rename('count').reset_index())
display(logs[logs.decision=='DENY']['user_id'].value_counts().head(5))
display(logs.groupby('hour')['decision'].value_counts().unstack(fill_value=0))
display(pd.crosstab(logs['user_id'], logs['resource_id']).head())
denies=logs[logs.decision=='DENY']['user_id'].value_counts().rename('denies').to_frame()
denies['z_score']=(denies['denies']-denies['denies'].mean())/(denies['denies'].std(ddof=0) or 1)
denies['anomaly']=denies['z_score'].abs()>2
display(denies.head(10))


### Visualisations

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,10))
logs.merge(users[['user_id','role']],on='user_id',how='left')['role'].value_counts().plot(kind='bar',ax=axes[0,0],title='Nombre d’accès par type utilisateur')
logs['decision'].value_counts().plot(kind='pie',autopct='%1.1f%%',ax=axes[0,1],title='Autorisations/refus'); axes[0,1].set_ylabel('')
sns.heatmap(pd.crosstab(logs['user_id'],logs['resource_type']),cmap='YlGnBu',ax=axes[1,0]); axes[1,0].set_title('Heatmap utilisateur vs ressource')
logs.set_index('timestamp').resample('D').size().plot(ax=axes[1,1],title='Évolution temporelle des accès')
plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); logs[logs.decision=='DENY']['user_id'].value_counts().head(5).sort_values().plot(kind='barh',color='#0f766e')
plt.title('Top 5 utilisateurs avec le plus de refus'); plt.xlabel('Nombre de refus'); plt.show()


## 6. Rapport de sécurité

In [ ]:
total=len(logs); deny_rate=logs['decision'].eq('DENY').mean()*100 if total else 0
top_user=logs[logs.decision=='DENY']['user_id'].value_counts().idxmax() if (logs.decision=='DENY').any() else 'aucun'
off_hours=logs[(logs['hour']<6)|(logs['hour']>20)].shape[0]
report=f'''# Rapport automatique

## Observations principales
1. Le journal contient {total} événements, taux de refus {deny_rate:.1f} %.
2. L'utilisateur avec le plus de refus est {top_user}.
3. {off_hours} accès sont hors plage 06h–20h.

## Recommandations
1. Renforcer la MFA sur les ressources sensibles/confidentielles.
2. Alerter sur les séries de refus et accès hors horaire.
3. Revoir régulièrement la matrice RBAC/ABAC.

## DICP / AAA
- Disponibilité: supervision des accès anormaux.
- Intégrité: séparation read/write/create.
- Confidentialité: MFA et ABAC départemental.
- Preuve: journalisation horodatée.
- AAA: authentification, autorisation, accounting.
'''
print(report)
